In [166]:
import xarray as xr
import json
import numpy as np
import os
import zarr

In [167]:
json_file_path = 'csmstate.json'

# Load JSON data
with open(json_file_path, 'r') as f:
    data = json.load(f)

In [168]:
# AI gen, tweaked
# Loops over all the key/values in the json dict and creates an array for each one 

# Create a zarr group
store = zarr.storage.LocalStore('csmstate.zarr')
root = zarr.group(store=store)

# Convert and store each field from the JSON
for key, value in data.items():
    if isinstance(value, (list, np.ndarray)):
        # Convert lists/arrays to numpy arrays and store
        arr = np.array(value)
        root.create_array(key, shape=arr.shape, chunks=arr.shape, dtype='f8')
        root[key] = arr
    elif isinstance(value, (int, float, str, bool)):
        # Store scalar values as arrays of size 1
        if (isinstance(value, int)):
            root.create_array(key, shape=1, chunks=1, dtype='i4')
        if (isinstance(value, float)):
            root.create_array(key, shape=1, chunks=1, dtype='f8')
        if (isinstance(value, str)):
            root.create_array(key, shape=1, chunks=1, dtype='S12')
        if (isinstance(value, bool)):
            root.create_array(key, shape=1, chunks=1, dtype='b1')
        root[key] = np.array([value])
    elif isinstance(value, dict):
        # Create a subgroup for nested dictionaries
        subgroup = root.create_group(key)
        for subkey, subvalue in value.items():
            if isinstance(subvalue, (list, np.ndarray)):
                arr = np.array(subvalue)
                subgroup.create_array(subkey, data=arr, shape=arr.shape, chunks=arr.shape)
            else:
                subgroup.create_array(subkey, data=np.array([subvalue]), shape=arr.shape, chunks=arr.shape)

print(f"Successfully converted csmstate.json to zarr format at {os.path.abspath('csmstate.zarr')}")

Successfully converted csmstate.json to zarr format at /Users/chkim/repos/code.usgs.gov/camerastatefile/notebooks/csmstate.zarr


/Users/chkim/miniforge3/envs/csmfile/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:99: UserWarning: The codec `vlen-bytes` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/Users/chkim/miniforge3/envs/csmfile/lib/python3.12/site-packages/zarr/core/array.py:3989: UserWarning: The dtype `|S12` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  meta = AsyncArray._create_metadata_v3(
/Users/chkim/miniforge3/envs/csmfile/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/Users/chkim/miniforge3/envs/csmfile/lib/python3.12/site-packages/zarr/core/a

In [169]:
root.tree()

/Users/chkim/miniforge3/envs/csmfile/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/Users/chkim/miniforge3/envs/csmfile/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/Users/chkim/miniforge3/envs/csmfile/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/Users/chkim/miniforge3/envs/csmfile/lib/python3.12/site-packages/zarr/co

/
├── m_centerEphemerisTime (1,) float64
├── m_covariance (256,) float64
├── m_currentParameterValue (16,) float64
├── m_detectorLineOrigin (1,) float64
├── m_detectorLineSumming (1,) float64
├── m_detectorSampleOrigin (1,) float64
├── m_detectorSampleSumming (1,) float64
├── m_distortionType (1,) int64
├── m_dtEphem (1,) float64
├── m_dtQuat (1,) float64
├── m_flyingHeight (1,) float64
├── m_focalLength (1,) float64
├── m_gsd (1,) float64
├── m_halfSwath (1,) float64
├── m_halfTime (1,) float64
├── m_iTransL (3,) float64
├── m_iTransS (3,) float64
├── m_ikCode (1,) int64
├── m_imageIdentifier (1,) StringDType()
├── m_intTimeLines (1,) float64
├── m_intTimeStartTimes (1,) float64
├── m_intTimes (1,) float64
├── m_majorAxis (1,) float64
├── m_maxElevation (1,) float64
├── m_minElevation (1,) float64
├── m_minorAxis (1,) float64
├── m_modelName (1,) StringDType()
├── m_nLines (1,) int64
├── m_nSamples (1,) int64
├── m_numPositions (1,) int64
├── m_numQuaternions (1,) int64
├── m_opticalDistCoeffs (3,) float64
├── m_parameterType (16,) int64
├── m_platformFlag (1,) int64
├── m_platformIdentifier (1,) StringDType()
├── m_positions (75,) float64
├── m_quaternions (1216,) float64
├── m_referencePointXyz (3,) float64
├── m_sensorIdentifier (1,) StringDType()
├── m_sensorName (1,) StringDType()
├── m_startingDetectorLine (1,) float64
├── m_startingDetectorSample (1,) float64
├── m_startingEphemerisTime (1,) float64
├── m_sunPosition (75,) float64
├── m_sunVelocity (75,) float64
├── m_t0Ephem (1,) float64
├── m_t0Quat (1,) float64
├── m_velocities (75,) float64
└── m_zDirection (1,) float64

In [170]:
root['m_quaternions']

<Array file://csmstate.zarr/m_quaternions shape=(1216,) dtype=float64>

In [156]:
# Create 2D array of positions and velocities
store2 = zarr.storage.LocalStore('csmstate2.zarr')
root2 = zarr.group(store=store2)

values = root2.create_array(name="values", shape=(2, 75), chunks=(75, 75), dtype="f4")
values[0] = np.array(data['m_positions'])
values[1] = np.array(data['m_velocities'])

In [159]:
values[:]

array([[ 8.64042000e+05,  3.35822450e+06,  1.20741525e+06,
         8.64240812e+05,  3.35644750e+06,  1.21232412e+06,
         8.64437562e+05,  3.35466400e+06,  1.21723062e+06,
         8.64632125e+05,  3.35287375e+06,  1.22213450e+06,
         8.64824688e+05,  3.35107675e+06,  1.22703600e+06,
         8.65015062e+05,  3.34927300e+06,  1.23193512e+06,
         8.65203312e+05,  3.34746250e+06,  1.23683162e+06,
         8.65389500e+05,  3.34564525e+06,  1.24172575e+06,
         8.65573562e+05,  3.34382150e+06,  1.24661725e+06,
         8.65755500e+05,  3.34199075e+06,  1.25150625e+06,
         8.65935312e+05,  3.34015350e+06,  1.25639288e+06,
         8.66113000e+05,  3.33830950e+06,  1.26127675e+06,
         8.66288500e+05,  3.33645875e+06,  1.26615825e+06,
         8.66461938e+05,  3.33460125e+06,  1.27103712e+06,
         8.66633250e+05,  3.33273725e+06,  1.27591350e+06,
         8.66802438e+05,  3.33086650e+06,  1.28078725e+06,
         8.66969500e+05,  3.32898900e+06,  1.28565838e+0

In [157]:
root2.tree()

/
└── values (2, 75) float32